# V031: protected short-window frontend on the Laplace body
One architecture change from V029. Objective/head/seed/features/data/fullrates are fixed. Two predeclared mixtures, unchanged practical gate; no automatic confirmation or submission.

In [ ]:
import os,json,hashlib,gc,time,fcntl
from pathlib import Path
from datetime import datetime,timezone
import numpy as np
import pandas as pd

def v31_require(c,m):
    if not bool(c):raise ValueError(m)
def v31_sha(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(1048576),b''):h.update(b)
    return h.hexdigest()
def v31_code(p):return '\n'.join(''.join(c['source']) for c in json.loads(Path(p).read_text())['cells'] if c['cell_type']=='code')
def v31_json(p,v,resume=False):
    p=Path(p)
    if p.exists():v31_require(resume and json.loads(p.read_text())==v,'Preserve existing immutable artifact: '+str(p));return
    p.parent.mkdir(parents=True,exist_ok=True);tmp=p.with_name(p.name+'.tmp');tmp.write_text(json.dumps(v,indent=2,allow_nan=False)+'\n');os.replace(tmp,p)
def v31_array(p,a):
    p=Path(p)
    if p.exists():v31_require(np.array_equal(np.load(p),a),'Existing array changed');return
    with p.with_name(p.name+'.tmp').open('wb') as f:np.save(f,a,allow_pickle=False)
    os.replace(p.with_name(p.name+'.tmp'),p)
def v31_event(out,event,**kw):
    with (out/'events.jsonl').open('a') as f:f.write(json.dumps(dict(event=event,time_utc=datetime.now(timezone.utc).isoformat(),**kw))+'\n')

def run_screen_v031(root='.',resume=False,*,expected_protocol_sha256,expected_source_code_sha256):
    root=Path(root).resolve();out=root/'outputs/controlled_experiments_v031';out.mkdir(parents=True,exist_ok=True);locks=[];started=time.monotonic()
    try:
        for p in [root/'outputs/.v011_gpu.lock',out/'.run.lock']:
            f=p.open('a+');fcntl.flock(f,fcntl.LOCK_EX|fcntl.LOCK_NB);locks.append(f)
        v31_require(v31_sha(root/'research/protocol_v031.json')==expected_protocol_sha256,'Frozen V031 protocol changed')
        plan=json.loads((root/'research/protocol_v031.json').read_text());v31_require(plan['status']=='frozen','Protocol not frozen')
        for p,h in plan['input_hashes'].items():v31_require(v31_sha(root/p)==h,'Input changed: '+p)
        for flag in ['PA3_RUN_V029','PA3_RUN_V030','PA3_RUN_V013','PA3_RUN_V014','PA3_RUN_V019','PA3_RUN_V021','PA3_RUN_V024','PA3_RUN_FINAL_V011','PA3_RUN_FINAL_V018']:v31_require(os.environ.get(flag,'0')!='1','Disable other run flags')
        namespace={'__name__':'ventilator_v031_components'}
        for n,h in plan['component_code_hashes'].items():
            text=v31_code(root/'research'/n);v31_require(hashlib.sha256(text.encode()).hexdigest()==h,'Component changed: '+n);exec(compile(text,n,'exec'),namespace)
        helper=v31_code(root/'controlled_experiments_v013.ipynb');v31_require(hashlib.sha256(helper.encode()).hexdigest()=='56cb9d7e824e58126bddd71c7c0f7063844974c31ee00a6dac23f2112657615a','Scoring helper changed');exec(compile(helper,'controlled_experiments_v013.ipynb','exec'),namespace)
        source_sha=hashlib.sha256(v31_code(root/'controlled_experiments_v031.ipynb').encode()).hexdigest();v31_require(source_sha==expected_source_code_sha256,'Driver source changed')
        sources=dict(plan['component_code_hashes'],**{'controlled_experiments_v031.ipynb':source_sha,'protocol_v031.json':expected_protocol_sha256,'controlled_experiments_v013.ipynb':'56cb9d7e824e58126bddd71c7c0f7063844974c31ee00a6dac23f2112657615a'})
        actual=namespace['V029_OBJECTIVE'];arm_recipe=plan['arm']
        v31_require(actual['loss_kind']==arm_recipe['loss_kind'] and actual['scale_floor_target_scaled']==arm_recipe['scale_floor'] and actual['initial_scale_target_scaled']==arm_recipe['scale_initialization'] and actual['curve_aux_weight']==arm_recipe['curve_aux_weight'],'Implemented objective constants differ from protocol')
        architecture=namespace['V031_ARCHITECTURE'];v31_require(architecture['span']==plan['arm']['adapter_span']==3 and architecture['offsets']==[-1,0,1] and architecture['activation']==plan['arm']['activation']=='SiLU' and architecture['protected_numeric_outputs']==plan['arm']['protected_numeric_outputs'] and architecture['protected_internal_joint_code']==plan['arm']['protected_internal_joint_code'],'Implemented adapter differs from protocol')
        wf=namespace['v031_workflow_namespace'](root)
        torch=wf['torch'];torch.set_num_threads(4);v31_require(torch.cuda.is_available(),'Actual course CUDA required');wf['seed_sequence'](plan['seed']);v31_require(torch.are_deterministic_algorithms_enabled(),'Strict deterministic execution required')
        for name in ['course_proofs.json','pruning_course_proofs.json']:
            proof=json.loads((out/name).read_text());v31_require(proof['status']=='passed' and proof['actual_cuda'] and all(proof['checks'].values()),'Actual CUDA proof failed: '+name)
            v31_require(proof['driver_source_code_sha256']==source_sha and proof['protocol_sha256']==expected_protocol_sha256,'CUDA proof not bound to current experiment')
        proof=json.loads((out/'course_proofs.json').read_text());v31_require(proof['exact_resume'] and proof['native_inference'] and proof['zero_adapter_control'],'Component CUDA requirements failed')
        pruningproof=json.loads((out/'pruning_course_proofs.json').read_text());v31_require(pruningproof['exact_pruned_resume'] and pruningproof['next_planned_experiment_executed'],'Pruning CUDA hook requirements failed')
        design=dict(protocol_sha256=expected_protocol_sha256,source_code_sha256=source_sha,source_fingerprints=sources,runtime=dict(torch=str(torch.__version__),cuda=torch.version.cuda,numpy=np.__version__,pandas=pd.__version__),strict_determinism=True)
        v31_json(out/'design.json',design,resume)
        old_csv=root/plan['preserved_official_CSV'];v31_require(v31_sha(old_csv)==plan['preserved_official_CSV_sha256'],'Official CSV changed')
        if (out/'screen_summary.json').exists():
            done=json.loads((out/'screen_summary.json').read_text());v31_require(resume and done['status']=='complete' and done['source_fingerprints']==sources,'Completed screen identity changed')
            for mname,rec in done['records'].items():
                v31_require(json.loads((out/mname/'record.json').read_text())==rec,'Completed report-record linkage changed')
                for key,hkey in [('checkpoint_file','checkpoint_sha256'),('last_checkpoint_file','last_checkpoint_sha256'),('report_file','report_sha256'),('prediction_file','prediction_sha256'),('scale_prediction_file','scale_prediction_sha256'),('training_outcome_file','training_outcome_sha256'),('RC_diagnostics_file','RC_diagnostics_sha256'),('position_diagnostics_file','position_diagnostics_sha256')]:v31_require(v31_sha(root/rec[key])==rec[hkey],'Completed model artifact changed: '+key)
                saved_outcome=json.loads((root/rec['training_outcome_file']).read_text());v31_require(saved_outcome['status']=='completed' and saved_outcome['source_fingerprints']==rec['training_source_fingerprints'] and saved_outcome['report']==json.loads((root/rec['report_file']).read_text()),'Completed effective training identity changed')
            for q in done['candidates']:v31_require(v31_sha(root/q['prediction_file'])==q['prediction_sha256'],'Completed mixture changed')
            return done
        frame=pd.read_csv(root/'data/train.csv',dtype=namespace['V013_DTYPES']);bids=wf['_validate_feature_frame_v005'](frame)
        labels=frame.pressure.to_numpy(copy=True).reshape(-1,80);phase=frame.u_out.to_numpy(copy=True).reshape(-1,80);ids=frame.id.to_numpy(copy=True).reshape(-1,80)
        rc=frame[['R','C']].to_numpy()[::80];known=frame.drop(columns='pressure');del frame;gc.collect()
        x,names,xbids=wf['make_sequence_features'](known,'combined');del known;gc.collect()
        v31_require(x.shape==(67905,80,47) and np.array_equal(xbids,bids),'Original full-feature schema changed')
        with np.load(root/'outputs/improvements_v005/sequence/split_breath_ids.npz') as z:
            fit=np.sort(pd.Index(bids).get_indexer(z['A_fit']));val=np.sort(pd.Index(bids).get_indexer(z['A_validation']))
        v31_require(len(fit)==61114 and len(val)==6791 and not np.intersect1d(fit,val).size and np.array_equal(np.sort(np.r_[fit,val]),np.arange(67905)),'Original A split changed')
        mask=phase[val]==0;truth=labels[val][mask].astype(np.float64);rows=ids[val][mask];counts=mask.sum(1)
        v31_require(len(truth)==206098 and np.array_equal(rows,np.load(root/'outputs/improvements_v005/sequence/A_validation_row_ids.npy')),'Original scored row alignment changed');v31_array(out/'A_row_ids.npy',rows)
        grid=namespace['pressure_codebook_v016'](labels[fit],phase[fit]);v31_require(np.array_equal(grid,np.load(root/'outputs/postprocessing_screen_v016/A_fit_pressure_codebook.npy')),'Fit-only pressure grid changed')
        predictions={k:np.load(root/p) for k,p in plan['reference_prediction_files'].items()}
        reference=namespace['selected_postprocessing_v016'](predictions,plan['reference_weights'],grid)
        v31_require(np.array_equal(reference,np.load(root/plan['reference_prediction_file'])) and abs(np.abs(reference-truth).mean()-plan['reference_A_MAE'])<1e-15,'Frozen reference recipe changed')
        arm=plan['arm'];name=arm['name'];stage=out/name;stage.mkdir(exist_ok=True);checkpoint=stage/'model.pt';report_path=stage/'report.json';factory=wf[arm['factory']]
        recipe=dict(model_spec=plan['base_model_spec'],train_cfg=plan['train_cfg'],seed=plan['seed'],feature_names=names,loss_kind=arm['loss_kind'],scale_floor=arm['scale_floor'],scale_initialization=arm['scale_initialization'],curve_aux_weight=arm['curve_aux_weight'],factory=arm['factory'],adapter_span=arm['adapter_span'],protected_numeric_outputs=arm['protected_numeric_outputs'],protected_internal_joint_code=arm['protected_internal_joint_code'],activation=arm['activation'])
        identity=dict(member=name,recipe=recipe,sources=sources,train_group_sha256=namespace['v013_ah'](bids[fit]),validation_group_sha256=namespace['v013_ah'](bids[val]));record_path=stage/'record.json'
        if record_path.exists():
            rec=json.loads(record_path.read_text());v31_require(resume and rec['identity']==identity,'Completed member identity changed')
            for key,hkey in [('checkpoint_file','checkpoint_sha256'),('last_checkpoint_file','last_checkpoint_sha256'),('report_file','report_sha256'),('prediction_file','prediction_sha256'),('scale_prediction_file','scale_prediction_sha256'),('training_outcome_file','training_outcome_sha256'),('RC_diagnostics_file','RC_diagnostics_sha256'),('position_diagnostics_file','position_diagnostics_sha256')]:v31_require(v31_sha(root/rec[key])==rec[hkey],'Completed member artifact changed: '+key)
            saved_outcome=json.loads((root/rec['training_outcome_file']).read_text());v31_require(saved_outcome['status']=='completed' and saved_outcome['source_fingerprints']==rec['training_source_fingerprints'] and saved_outcome['report']==json.loads(report_path.read_text()),'Completed effective training identity changed')
            v31_require(rec['training_source_fingerprints']==wf['bind_pruning_sources_v025'](root,root/plan['pruning']['policy_file'],sources),'Completed source bindings differ')
        else:
            v31_event(out,'A_fit_started',member=name,seed=plan['seed'],objective=arm['loss_kind'])
            args=(x[fit],labels[fit],phase[fit],x[val],labels[val],phase[val],plan['base_model_spec'],plan['train_cfg'],plan['seed'],checkpoint)
            common=dict(epoch_log_path=stage/'epochs.jsonl',feature_names=names,device='cuda',resume=resume and wf['_last_path'](checkpoint).exists(),arm='v031_A_'+name,source_fingerprints=sources,model_factory=factory,loss_kind=arm['loss_kind'])
            jobs=[dict(id='v031_A_'+name,variant='conv3_laplace_curve',fit_args=args,fit_kwargs=common)]
            outcomes=wf['run_training_queue_v025'](jobs,project_root=root,policy_path=root/plan['pruning']['policy_file'],expected_policy_sha256=plan['pruning']['policy_sha256'],state_path=out/'training_queue.json')
            outcome=outcomes[0];v31_require(outcome['status']=='completed' and outcome['report']['executed_epochs']==200,'Representative fit incomplete; preserve exact state for resume')
            report=outcome['report'];v31_json(report_path,report,resume)
            mu,scale=wf['predict_checkpoint_laplace_v031'](checkpoint,x[val],device='cuda',batch_size=512,model_factory=factory,expected_arm='v031_A_'+name,source_fingerprints=outcome['source_fingerprints'],return_scale=True)
            v31_require(mu.shape==scale.shape==(6791,80) and np.isfinite(mu).all() and np.isfinite(scale).all() and (scale>0).all(),'Native mean/scale predictions invalid')
            p=mu[mask].astype(np.float64);v31_array(stage/'A_predictions.npy',p);v31_array(stage/'A_scale_predictions.npy',scale[mask].astype(np.float64))
            v31_require(abs(np.abs(p-truth).mean()-report['validation_mae'])<1e-12,'Native best-checkpoint score differs')
            native_error=np.abs(mu.astype(np.float64)-labels[val].astype(np.float64));native_total_error=float(native_error[mask].sum());diag=[]
            for setting in sorted(set(map(tuple,rc[val]))):
                select=np.all(rc[val]==setting,axis=1);sel=mask[select];err=native_error[select][sel];s=scale[select][sel].astype(np.float64)
                diag.append(dict(R=int(setting[0]),C=int(setting[1]),points=len(err),MAE=float(err.mean()),total_absolute_error=float(err.sum()),fraction_total_absolute_error=float(err.sum()/native_total_error),native_scale_mean=float(s.mean()),native_scale_median=float(np.median(s)),native_scale_p90=float(np.quantile(s,.9)),native_scale_min=float(s.min()),mean_inverse_native_scale=float((1./s).mean())))
            v31_json(stage/'validation_scale_by_RC.json',diag,resume)
            position_diag=[]
            for pos in range(80):
                select=mask[:,pos]
                if not select.any():continue
                err=native_error[select,pos];s=scale[select,pos].astype(np.float64)
                position_diag.append(dict(position=pos,points=len(err),MAE=float(err.mean()),total_absolute_error=float(err.sum()),fraction_total_absolute_error=float(err.sum()/native_total_error),native_scale_mean=float(s.mean()),native_scale_median=float(np.median(s)),native_scale_p90=float(np.quantile(s,.9)),native_scale_min=float(s.min()),mean_inverse_native_scale=float((1./s).mean())))
            v31_json(stage/'validation_scale_by_position.json',position_diag,resume)
            rec=dict(identity=identity,identity_sha256=namespace['v013_sig'](identity),checkpoint_file=str(checkpoint.relative_to(root)),checkpoint_sha256=v31_sha(checkpoint),report_file=str(report_path.relative_to(root)),report_sha256=v31_sha(report_path),prediction_file=str((stage/'A_predictions.npy').relative_to(root)),prediction_sha256=v31_sha(stage/'A_predictions.npy'),scale_prediction_file=str((stage/'A_scale_predictions.npy').relative_to(root)),scale_prediction_sha256=v31_sha(stage/'A_scale_predictions.npy'),prediction_array_sha256=namespace['v013_ah'](p),row_ids_array_sha256=namespace['v013_ah'](rows),native_A_MAE=float(np.abs(p-truth).mean()),best_epoch=report['best_epoch'],selected_lr_schedule=report['selected_lr_schedule'],executed_epochs=report['executed_epochs'],training_breaths=61114,elapsed_seconds=report['elapsed_seconds'],training_outcome_file=str(checkpoint.with_name(checkpoint.stem+'.outcome.json').relative_to(root)),training_source_fingerprints=outcome['source_fingerprints'],pruning_status=outcome['status'])
            rec.update(last_checkpoint_file=str(wf['_last_path'](checkpoint).relative_to(root)),last_checkpoint_sha256=v31_sha(wf['_last_path'](checkpoint)),training_outcome_sha256=v31_sha(root/rec['training_outcome_file']),RC_diagnostics_file=str((stage/'validation_scale_by_RC.json').relative_to(root)),RC_diagnostics_sha256=v31_sha(stage/'validation_scale_by_RC.json'),position_diagnostics_file=str((stage/'validation_scale_by_position.json').relative_to(root)),position_diagnostics_sha256=v31_sha(stage/'validation_scale_by_position.json'))
            v31_json(record_path,rec)
        predictions[name]=np.load(root/rec['prediction_file']);candidates=[]
        for cname,weights in plan['candidates'].items():
            p=namespace['selected_postprocessing_v016']({k:predictions[k] for k in weights},weights,grid);error=np.abs(p-truth);interval=namespace['v013_interval'](error,np.abs(reference-truth),counts,plan['bootstrap_seed']);gain=float(np.abs(reference-truth).mean()-error.mean());path=out/(cname+'_A_predictions.npy');v31_array(path,p)
            candidates.append(dict(name=cname,weights=weights,MAE=float(error.mean()),gain=gain,interval=interval,eligible=bool(gain>=plan['minimum_A_gain'] and interval['high']<0),prediction_file=str(path.relative_to(root)),prediction_sha256=v31_sha(path)))
        eligible=[q for q in candidates if q['eligible']];selected=min(eligible,key=lambda q:(q['MAE'],q['name'])) if eligible else None
        result=dict(status='complete',protocol_sha256=expected_protocol_sha256,source_code_sha256=source_sha,source_fingerprints=sources,reference_A_MAE=plan['reference_A_MAE'],records={name:rec},candidates=candidates,selected_for_confirmation=selected,no_F_fit_performed=True,no_final_fit_performed=True,no_submission=True,elapsed_seconds=time.monotonic()-started,completed_utc=datetime.now(timezone.utc).isoformat())
        v31_require(v31_sha(old_csv)==plan['preserved_official_CSV_sha256'],'Accepted CSV changed during screen');v31_json(out/'screen_summary.json',result);return result
    finally:
        for f in reversed(locks):fcntl.flock(f,fcntl.LOCK_UN);f.close()

if os.environ.get('PA3_RUN_V031','0')=='1':
    V031_PINS=json.loads((Path.cwd()/'research/freeze_v031.json').read_text())
    V031_SCREEN_RESULT=run_screen_v031(Path.cwd(),resume=os.environ.get('PA3_RESUME_V031','0')=='1',expected_protocol_sha256=V031_PINS['protocol_sha256'],expected_source_code_sha256=V031_PINS['source_code_sha256'])
    print(json.dumps(V031_SCREEN_RESULT['candidates'],indent=2))
